In [1]:
from google.colab import files
uploaded = files.upload()


Saving comm_log.db to comm_log.db


In [2]:
!ls -la

total 28
drwxr-xr-x 1 root root  4096 Sep 12 06:02 .
drwxr-xr-x 1 root root  4096 Sep 12 05:49 ..
-rw-r--r-- 1 root root 12288 Sep 12 06:02 comm_log.db
drwxr-xr-x 4 root root  4096 Sep  4 13:25 .config
drwxr-xr-x 1 root root  4096 Sep  4 13:25 sample_data


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('comm_log.db')

In [4]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,campaign
1,communication_log


In [5]:
query = """
SELECT COUNT(*) AS naive_count
FROM communication_log
WHERE merchant_id = 501;
"""
pd.read_sql_query(query, conn)

,naive_count
0,30


In [6]:
query = """
SELECT c.creation_status, c.processing_status, COUNT(*) AS row_count
FROM communication_log cl
JOIN campaign c ON cl.communication_id = c.id
WHERE cl.merchant_id = 501
GROUP BY c.creation_status, c.processing_status;
"""
pd.read_sql_query(query, conn)

,creation_status,processing_status,row_count
0,approval_awaiting,processed,4
1,approved,processed,26


In [7]:
query = """
SELECT COUNT(*) AS after_status_filter
FROM communication_log cl
JOIN campaign c ON cl.communication_id = c.id
WHERE cl.merchant_id = 501
  AND c.creation_status = 'approved'
  AND c.processing_status = 'processed';
"""
pd.read_sql_query(query, conn)

,after_status_filter
0,26


In [8]:
query = """
SELECT id, parent_id, name
FROM campaign
WHERE merchant_id = 501
ORDER BY id;
"""
pd.read_sql_query(query, conn)

,id,parent_id,name
0,9001,NaN,Diwali Cart Recovery - Wave 1
1,9002,9001.0,Diwali Cart Recovery - Retry A
2,9003,9002.0,Diwali Cart Recovery - Retry B
3,9004,9001.0,Diwali Cart Recovery - Retry C (pending)
4,9101,NaN,Diwali Flash Sale - Standalone
5,9201,NaN,Diwali Wave 2
6,9202,9201.0,Diwali Wave 2 - Retry


In [9]:
query = """
WITH RECURSIVE chain AS (
  -- base case: every campaign with no parent is a root of its own chain
  SELECT id, parent_id, id AS root_id
  FROM campaign
  WHERE parent_id IS NULL

  UNION ALL

  -- recursive case: any campaign whose parent is already in `chain`
  -- inherits that parent's root_id
  SELECT c.id, c.parent_id, chain.root_id
  FROM campaign c
  JOIN chain ON c.parent_id = chain.id
)
SELECT * FROM chain
ORDER BY root_id, id;
"""
pd.read_sql_query(query, conn)

,id,parent_id,root_id
0,9001,NaN,9001
1,9002,9001.0,9001
2,9003,9002.0,9001
3,9004,9001.0,9001
4,9101,NaN,9101
5,9201,NaN,9201
6,9202,9201.0,9201


In [10]:
query = """
WITH RECURSIVE chain AS (
  SELECT id, parent_id, id AS root_id
  FROM campaign
  WHERE parent_id IS NULL
  UNION ALL
  SELECT c.id, c.parent_id, chain.root_id
  FROM campaign c
  JOIN chain ON c.parent_id = chain.id
)
SELECT root_id, COUNT(*) AS chain_size
FROM chain
GROUP BY root_id;
"""
pd.read_sql_query(query, conn)

,root_id,chain_size
0,9001,4
1,9101,1
2,9201,2


In [11]:
query = """
WITH RECURSIVE chain AS (
  SELECT id, parent_id, id AS root_id
  FROM campaign
  WHERE parent_id IS NULL
  UNION ALL
  SELECT c.id, c.parent_id, chain.root_id
  FROM campaign c
  JOIN chain ON c.parent_id = chain.id
),
chain_sizes AS (
  SELECT root_id, COUNT(*) AS chain_size
  FROM chain
  GROUP BY root_id
),
eligible_logs AS (
  SELECT
    cl.customer_id,
    cl.delivery_status,
    ch.root_id,
    cs.chain_size
  FROM communication_log cl
  JOIN campaign c ON cl.communication_id = c.id
  JOIN chain ch ON c.id = ch.id
  JOIN chain_sizes cs ON ch.root_id = cs.root_id
  WHERE cl.merchant_id = 501
    AND c.creation_status = 'approved'
    AND c.processing_status = 'processed'
)
SELECT * FROM eligible_logs
ORDER BY root_id, customer_id;
"""
pd.read_sql_query(query, conn)

,customer_id,delivery_status,root_id,chain_size
0,C1,900,9001,4
1,C10,900,9001,4
2,C2,1100,9001,4
3,C2,900,9001,4
4,C3,1100,9001,4
5,C3,1100,9001,4
6,C3,900,9001,4
7,C4,900,9001,4
8,C5,900,9001,4
9,C6,900,9001,4


In [12]:
query = """
WITH RECURSIVE chain AS (
  SELECT id, parent_id, id AS root_id
  FROM campaign
  WHERE parent_id IS NULL
  UNION ALL
  SELECT c.id, c.parent_id, chain.root_id
  FROM campaign c
  JOIN chain ON c.parent_id = chain.id
),
chain_sizes AS (
  SELECT root_id, COUNT(*) AS chain_size
  FROM chain
  GROUP BY root_id
),
eligible_logs AS (
  SELECT
    cl.customer_id,
    cl.delivery_status,
    ch.root_id,
    cs.chain_size
  FROM communication_log cl
  JOIN campaign c ON cl.communication_id = c.id
  JOIN chain ch ON c.id = ch.id
  JOIN chain_sizes cs ON ch.root_id = cs.root_id
  WHERE cl.merchant_id = 501
    AND c.creation_status = 'approved'
    AND c.processing_status = 'processed'
    AND cl.delivery_status = 900
)
SELECT
  (SELECT COUNT(*) FROM eligible_logs WHERE chain_size = 1) AS standalone_rows,
  (SELECT COUNT(DISTINCT root_id || '-' || customer_id) FROM eligible_logs WHERE chain_size > 1) AS chain_distinct_customers,
  (SELECT COUNT(*) FROM eligible_logs WHERE chain_size = 1)
    + (SELECT COUNT(DISTINCT root_id || '-' || customer_id) FROM eligible_logs WHERE chain_size > 1) AS target_base;
"""
pd.read_sql_query(query, conn)

,standalone_rows,chain_distinct_customers,target_base
0,7,15,22
